# Expanded-Label Geopolitical Exposure Event Study
## V2 evaluation of frozen text and embedding signals
This notebook evaluates the expanded text-label propagation against market reactions surrounding Russia's invasion of Ukraine. It preserves the original V1 event study and uses only the frozen V2 scores created before outcomes were loaded into the V2 propagation notebook. Because the same event was examined in V1, this remains an expanded-sample robustness exercise rather than an independent confirmation.


## 1. Verify frozen V2 inputs
Recompute every artifact hash before opening event outcomes.


In [ ]:
from pathlib import Path
import hashlib,importlib,json,sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='geopolitical_exposure': GEO_ROOT,ROOT=ROOT,ROOT.parent
else: GEO_ROOT=ROOT/'geopolitical_exposure'
if not GEO_ROOT.exists(): raise FileNotFoundError('Open from Asset_Embeddings or geopolitical_exposure.')
sys.path.insert(0,str(ROOT/'.packages'));sys.path.insert(0,str(GEO_ROOT/'src'))
import event_study,expanded_event_evaluation
importlib.reload(event_study);importlib.reload(expanded_event_evaluation)
from event_study import industry_seed_scores
from expanded_event_evaluation import direct_seed_reactions,evaluate_model_scores,evaluate_placebos
EXPANSION=GEO_ROOT/'embedding_expansion_v2';EVENT_DATA=GEO_ROOT/'event_data';RESULTS=GEO_ROOT/'event_study_v2';MANIFESTS=GEO_ROOT/'manifests';RESULTS.mkdir(parents=True,exist_ok=True)
freeze=json.loads((MANIFESTS/'embedding_expansion_freeze_v2.json').read_text());checks=[]
for artifact in freeze['artifacts']:
    path=EXPANSION/f"{artifact['artifact']}.parquet";checks.append({'artifact':artifact['artifact'],'exists':path.exists(),'hash_matches':path.exists() and hashlib.sha256(path.read_bytes()).hexdigest()==artifact['sha256'],'bytes_match':path.exists() and path.stat().st_size==artifact['bytes']})
freeze_check=pd.DataFrame(checks);display(freeze_check)
assert not freeze['outcomes_loaded'] and freeze_check[['exists','hash_matches','bytes_match']].all().all()


## 2. Freeze the V2 evaluation protocol
The model grid, event windows, controls, sample exclusions, and interpretation are fixed before loading CARs.


In [ ]:
protocol={'event_id':'russia_ukraine_invasion_2022','event_date':'2022-02-24','embedding_snapshot':'2021Q4','outcome_windows':[[0,1],[0,5],[1,20]],'expected_return_model':'firm market model using CRSP value-weighted market return','primary_sample':'common 1,805-firm universe excluding all 89 directional V2 seed firms','models':['five holdings embeddings','characteristics similarity','industry similarity','direct text descriptive benchmark','100 shuffled-seed all-S12 placebos'],'primary_tests':['Spearman score-CAR correlation','top-minus-bottom score quintile CAR','incremental adjusted R2 beyond frozen characteristics and SIC2 fixed effects'],'multiple_testing':'Benjamini-Hochberg q-values reported across the transparent model/channel/window grid','timing_interpretation':'researcher-information association test; not a tradable strategy','confirmatory_status':'expanded-sample robustness after V1 outcomes were previously inspected; separate held-out event required for independent confirmation','expansion_manifest_sha256':freeze['sha256']}
protocol['sha256']=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest();(MANIFESTS/'event_evaluation_protocol_v2.json').write_text(json.dumps(protocol,indent=2));display(pd.DataFrame([protocol]).T.rename(columns={0:'frozen_value'}))


## 3. Load checkpointed event outcomes
Reuse the immutable CRSP market-model CARs already constructed in the original event study. No WRDS connection is needed.


In [ ]:
required={'cars':EVENT_DATA/'market_model_cars.parquet','industry':EVENT_DATA/'crsp_point_in_time_industry_2021q4.parquet'}
missing=[name for name,path in required.items() if not path.exists()]
if missing: raise FileNotFoundError(f'Missing event-data checkpoints: {missing}. Run the original event-study acquisition stages.')
cars=pd.read_parquet(required['cars']);industry=pd.read_parquet(required['industry']);car_columns=['car_0_1','car_0_5','car_1_20']
display(pd.DataFrame([{'firms_with_market_model':len(cars),'industry_firms':industry.permco.nunique(),**{f'{c}_coverage':cars[c].notna().mean() for c in car_columns}}]));display(cars[car_columns].describe(percentiles=[.01,.1,.5,.9,.99]).T)


## 4. Build and lock the common non-seed evaluation panel
All model representations use the same firms and the same exclusion of every directional text seed.


In [ ]:
seeds=pd.read_parquet(EXPANSION/'directional_firm_channel_seeds.parquet');embedding_scores=pd.read_parquet(EXPANSION/'embedding_propagated_scores.parquet');characteristic_scores=pd.read_parquet(EXPANSION/'characteristic_propagated_scores.parquet')
universe=embedding_scores[embedding_scores.representation.eq('all_s12')][['permco','firm_name']].drop_duplicates();universe_industry=universe.merge(industry[['permco','siccd','sic2']],on='permco',how='left',validate='one_to_one')
industry_scores=industry_seed_scores(universe_industry,seeds,'sic2');industry_scores.to_parquet(RESULTS/'industry_seed_scores.parquet',index=False)
all_seed_firms=set(seeds.firm_id.astype(int));model_scores=pd.concat([embedding_scores,characteristic_scores,industry_scores],ignore_index=True,sort=False);evaluation=model_scores.loc[~model_scores.permco.isin(all_seed_firms)].merge(cars,on='permco',how='inner',validate='many_to_one')
evaluation.to_parquet(RESULTS/'locked_nonseed_evaluation_panel.parquet',index=False)
sample_audit=evaluation.groupby(['representation','channel'],as_index=False).agg(firms=('permco','nunique'));display(sample_audit);assert sample_audit.firms.nunique()==1


## 5. Direct-text descriptive benchmark
Relate signed text scores to CARs only among directly labeled firms. This is descriptive and not directly comparable to non-seed propagation performance.


In [ ]:
direct_panel,direct_summary=direct_seed_reactions(seeds,cars,car_columns);direct_panel.to_parquet(RESULTS/'direct_seed_event_reactions.parquet',index=False);direct_summary.to_parquet(RESULTS/'direct_seed_event_summary.parquet',index=False);display(direct_summary)


## 6. Primary non-seed model comparison
Estimate rank correlations, quintile spreads, and incremental regressions controlling for pre-event characteristics and SIC2 fixed effects.


In [ ]:
controls=pd.read_parquet(ROOT/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet');control_columns=['beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1'];firm_controls=universe_industry.merge(controls,on='permco',how='left',validate='one_to_one')
model_results=evaluate_model_scores(evaluation,firm_controls,control_columns,car_columns);model_results.to_parquet(RESULTS/'locked_model_comparison.parquet',index=False)
display(model_results.sort_values(['outcome','channel','incremental_adjusted_r2'],ascending=[True,True,False]))


## 7. All-S12 shuffled-seed placebo test
Compare each actual all-S12 rank correlation with the frozen 100-repetition null distribution.


In [ ]:
placebos=pd.read_parquet(EXPANSION/'shuffled_embedding_placebos_100.parquet');nonseed_ids=set(universe.permco)-set(seeds.firm_id);placebos=placebos[placebos.permco.isin(nonseed_ids)]
placebo_correlations,placebo_summary=evaluate_placebos(placebos,cars,car_columns,model_results);placebo_correlations.to_parquet(RESULTS/'placebo_correlations_100.parquet',index=False);placebo_summary.to_parquet(RESULTS/'all_s12_placebo_summary.parquet',index=False);display(placebo_summary)


## 8. Compare V1 and V2 event-study estimates
Report whether the expanded labels preserve, weaken, or reverse the original all-S12 associations for the three shared channels.


In [ ]:
v1_path=GEO_ROOT/'event_study/locked_model_comparison.parquet';v1=pd.read_parquet(v1_path);shared=sorted(set(v1.channel)&set(model_results.channel));v1_all=v1[(v1.representation=='all_s12')&v1.channel.isin(shared)];v2_all=model_results[(model_results.representation=='all_s12')&model_results.channel.isin(shared)]
comparison=v1_all.merge(v2_all,on=['representation','channel','outcome'],suffixes=('_v1','_v2'),validate='one_to_one');comparison['rho_change_v2_minus_v1']=comparison.spearman_rho_v2-comparison.spearman_rho_v1;comparison['incremental_r2_change_v2_minus_v1']=comparison.incremental_adjusted_r2_v2-comparison.incremental_adjusted_r2_v1;comparison.to_parquet(RESULTS/'v1_v2_event_comparison.parquet',index=False)
display(comparison[['channel','outcome','spearman_rho_v1','spearman_rho_v2','rho_change_v2_minus_v1','incremental_adjusted_r2_v1','incremental_adjusted_r2_v2','incremental_r2_change_v2_minus_v1']])


## 9. Decision table and transparent interpretation limits
Summarize the strongest results using predeclared thresholds while retaining the complete grid. Statistical significance alone does not establish economic validity.


In [ ]:
decision=model_results.copy();decision['nominal_rank_signal']=decision.spearman_pvalue.lt(.05);decision['fdr_rank_signal']=decision.spearman_bh_qvalue.lt(.10);decision['fdr_incremental_signal']=decision.score_hc3_bh_qvalue.lt(.10);decision['positive_incremental_r2']=decision.incremental_adjusted_r2.gt(0);decision['passes_joint_screen']=decision.fdr_rank_signal & decision.fdr_incremental_signal & decision.positive_incremental_r2
decision.to_parquet(RESULTS/'model_decision_table.parquet',index=False);display(decision.sort_values(['passes_joint_screen','spearman_bh_qvalue','incremental_adjusted_r2'],ascending=[False,True,False]).head(40))


## 10. Freeze evaluated outputs
Hash every evaluated artifact and record the exploratory status of this same-event expanded-sample exercise.


In [ ]:
artifact_names=['industry_seed_scores.parquet','locked_nonseed_evaluation_panel.parquet','direct_seed_event_reactions.parquet','direct_seed_event_summary.parquet','locked_model_comparison.parquet','placebo_correlations_100.parquet','all_s12_placebo_summary.parquet','v1_v2_event_comparison.parquet','model_decision_table.parquet'];inventory=[]
for name in artifact_names:
    path=RESULTS/name;inventory.append({'artifact':path.stem,'rows':len(pd.read_parquet(path)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
inventory=pd.DataFrame(inventory);inventory.to_parquet(RESULTS/'output_inventory.parquet',index=False)
manifest={'stage':'expanded_label_event_study_v2','protocol_sha256':protocol['sha256'],'expansion_manifest_sha256':freeze['sha256'],'event_outcomes_opened':True,'confirmatory_status':protocol['confirmatory_status'],'seed_firms':int(seeds.firm_id.nunique()),'channels':sorted(seeds.channel.unique()),'artifacts':inventory.to_dict('records')}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest();(MANIFESTS/'event_study_evaluated_v2.json').write_text(json.dumps(manifest,indent=2));display(inventory);print('V2 evaluated manifest:',manifest['sha256'])
